# PromptShield — benign hard-negative mitigation (Colab, single pre-registered run)

**Status: preparation only. Training has NOT run. Nothing in this notebook runs until you execute the cells, top to bottom, yourself.**

**Environment guard build: `hardneg-env-v3` (2026-10-26).** Cell 2 and Cell 3 must print this exact build; Cells 4 and 5 refuse to run without it in the same runtime.

This notebook executes the frozen experiment defined in `docs/benign-hard-negative-mitigation-plan.md`
via the existing module `src/promptshield/hard_negative_experiment.py` — the *same* gates, dataset
selection, hyperparameters, threshold (0.90), gateway policy (`band_v1`), and acceptance criteria
(G1–G6) that are used locally. Nothing here is a new experiment.

**Hard rules (enforced by the cells below):**

1. `test.parquet` is **never transferred, opened, or evaluated** — the bundle does not contain it and cell 1 verifies its absence.
2. **Exactly one training run.** If any cell fails or a gate blocks, **stop**, record the output, and report back — do not rerun, tune, or change thresholds.
3. Use a **CPU runtime** (Runtime → Change runtime type → CPU). The frozen plan trains on CPU; a GPU would change the baseline-reproduction gate.
4. Only the files listed in `docs/hard-negative-colab-runbook.md` are transferred. **No repository, no `.git`, no credentials, no secrets.**
5. **No downloads** of external models or datasets. The notebook forces Hugging Face offline mode; the model comes from the uploaded frozen checkpoint.
6. Reports and console output are **aggregate metrics and metadata only** — no raw prompts are printed or written.
7. The frozen baseline checkpoint is verified by hash **before and after** the experiment and is never written to.
8. Colab's optional `torchvision`/`torchaudio` packages are removed before and after the pinned CPU install, then verified absent in Cell 2 and again in Cell 3. PromptShield does not use them, and their preinstalled builds conflict with `torch==2.6.0+cpu`.
9. Cell 3 prints `guard build hardneg-env-v3` and checks distribution metadata, loaded submodules such as `torchvision._C`, Transformers' cached availability flags, `transformers.image_utils`, and the concrete DeBERTa classes. Cells 4 and 5 refuse to run unless that same guard completed in the same runtime.

Estimated time: ~15–35 minutes on a standard Colab CPU runtime (baseline audit, 3-epoch training on 414 rows, candidate audit).

In [ ]:
# CONFIG — only these paths are ever used.
from pathlib import Path

WORKDIR = Path('/content/promptshield')   # mirrors the repo layout; do not change
BUNDLE_ZIP = Path('/content/drive/MyDrive/Colab Notebooks/promptshield-colab-bundle.zip')  # adjust if stored elsewhere on Drive

# Frozen expectations (aggregate metadata only: hashes, never prompt text).
EXPECTED_TRAIN_SHA256 = 'c0724f8e00eebb65c001be9ba94291d1608e5da0374012e53dfd4077c5f92558'
EXPECTED_VALIDATION_SHA256 = 'bbd417bfd0df1553dd769e814a0b62ca58ad5b1d08d153c90f66937a4af74deb'
EXPECTED_BASELINE_FILES = {  # models/deberta-v3-small, frozen audit 2026-10-17
    'added_tokens.json': 'dc046d04c9b0ada7ae6f1dc89c465801799acdf0c9a6aab8c15a1b2d5ca4e91f',
    'config.json': '44e8fe36a9ea5659b039545691a0896e21ab5629f37b2c7f4480354433902f0d',
    'model.safetensors': '2cefb8e40ade17940899ef523c7dea2a1bd1813ba0373edda84481a32c783614',
    'special_tokens_map.json': '9463f61e1b109a8eb4688b829260d7c6b1e6dff04c98ff7269bb89e2b92369b9',
    'spm.model': 'c679fbf93643d19aab7ee10c0b99e460bdbc02fedf34b92b05af343b4af586fd',
    'tokenizer.json': '37fe3823ad5737f0033c832905c8d4089ed4008da79682ab3fd85c8ce8e5b040',
    'tokenizer_config.json': 'e6c4c771911c211618a2d46488dc7e9e499f051773cd4ff4faeca6fe55fdf569',
}
print('config ready; WORKDIR =', WORKDIR)

In [ ]:
# CELL 1 — unpack the exact-file bundle and verify layout, hashes, and test-split absence.
# No model is loaded here, nothing is trained, nothing is written outside WORKDIR.
import hashlib
import shutil
import zipfile
from pathlib import PurePosixPath

assert 'google.colab' in __import__('sys').modules, 'This notebook is intended for Google Colab only.'

# Mount Google Drive first (approve the authorization pop-up) so the bundle zip can be found.
if str(BUNDLE_ZIP).startswith('/content/drive') and not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

# The archive may contain only direct .py modules of src/promptshield plus the
# exact data/model files below. This is checked BEFORE anything is written.
APPROVED_DATA_FILES = {
    'data/raw/train.parquet',
    'data/raw/validation.parquet',
    'data/raw-eval/dolly/databricks-dolly-15k.jsonl',
    'data/raw-eval/oasst1/train.parquet',
}
APPROVED_BASELINE_FILES = set(EXPECTED_BASELINE_FILES)
APPROVED_SCRIPT_FILES = {'scripts/fp_benign_diagnostic.py'}
APPROVED_DIR_PREFIXES = {
    'data/raw',
    'data/raw-eval',
    'models/deberta-v3-small',
    'scripts',
    'src/promptshield',
}

def approved_file(name):
    path = PurePosixPath(name)
    if name in APPROVED_DATA_FILES or name in APPROVED_SCRIPT_FILES:
        return True
    if path.parent.as_posix() == 'models/deberta-v3-small' and path.name in APPROVED_BASELINE_FILES:
        return True
    return (
        len(path.parts) == 3
        and path.parts[0] == 'src'
        and path.parts[1] == 'promptshield'
        and path.suffix == '.py'
    )

def normalize_member(name):
    name = name.replace('\\', '/')
    path = PurePosixPath(name)
    parts = path.parts
    assert name and not path.is_absolute(), f'unsafe archive path: {name!r}'
    assert not (parts and ':' in parts[0]), f'unsafe archive path: {name!r}'
    assert '..' not in parts, f'unsafe archive path: {name!r}'
    return path.as_posix()

def extract_verified_bundle():
    with zipfile.ZipFile(BUNDLE_ZIP) as archive:
        members = []
        seen = set()
        for info in archive.infolist():
            name = normalize_member(info.orig_filename)
            if name.endswith('/'):
                prefix = name.rstrip('/')
                assert prefix in APPROVED_DIR_PREFIXES, f'unexpected archive directory: {name}'
                continue
            assert PurePosixPath(name).name != 'test.parquet', 'test.parquet must never be transferred'
            assert '.git' not in PurePosixPath(name).parts, 'Repository metadata must never be transferred.'
            assert approved_file(name), f'unexpected archive member: {name}'
            assert name not in seen, f'duplicate archive member: {name}'
            seen.add(name)
            members.append((info, name))
        assert members, 'bundle contains no files'
        for info, name in members:
            destination = WORKDIR.joinpath(*PurePosixPath(name).parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, open(destination, 'wb') as target:
                shutil.copyfileobj(source, target, length=1 << 20)

if not (WORKDIR / 'src' / 'promptshield' / 'hard_negative_experiment.py').exists():
    assert BUNDLE_ZIP.is_file(), f'Bundle not found: {BUNDLE_ZIP}. Mount Drive or fix BUNDLE_ZIP.'
    extract_verified_bundle()

required = [
    'src/promptshield/__init__.py',
    'src/promptshield/hard_negative_experiment.py',
    'src/promptshield/deberta_decision_audit.py',
    'src/promptshield/inference.py',
    'src/promptshield/benign_chat_eval.py',
    'src/promptshield/review_suggestions.py',
    'src/promptshield/error_review.py',
    'src/promptshield/gateway.py',
    'src/promptshield/baseline.py',
    'data/raw/train.parquet',
    'data/raw/validation.parquet',
    'data/raw-eval/dolly/databricks-dolly-15k.jsonl',
    'data/raw-eval/oasst1/train.parquet',
    'models/deberta-v3-small/model.safetensors',
    'scripts/fp_benign_diagnostic.py',
]
missing = [item for item in required if not (WORKDIR / item).exists()]
assert not missing, f'Missing required files: {missing}'

# Safety: the test split must not even exist in this runtime.
test_files = [str(p) for p in WORKDIR.rglob('test.parquet')]
assert not test_files, f'test.parquet must never be transferred: {test_files}'
assert not (WORKDIR / '.git').exists(), 'Repository metadata must never be transferred.'

assert sha256(WORKDIR / 'data/raw/train.parquet') == EXPECTED_TRAIN_SHA256, 'train.parquet hash mismatch'
assert sha256(WORKDIR / 'data/raw/validation.parquet') == EXPECTED_VALIDATION_SHA256, 'validation.parquet hash mismatch'
for name, expected in EXPECTED_BASELINE_FILES.items():
    actual = sha256(WORKDIR / 'models/deberta-v3-small' / name)
    assert actual == expected, f'baseline file corrupted in transfer: {name}'

print('bundle verified: layout complete, baseline checkpoint hash-verified, test split absent (aggregate metadata only)')

In [ ]:
# CELL 2 — pinned dependencies matching the frozen audit environment (report runtime: torch 2.6.0+cpu,
# transformers 4.49.0, sentencepiece 0.2.1, pyarrow 22.0.0, scikit-learn 1.8.0, joblib 1.5.3).
# CPU-only torch wheel: no CUDA stack, no GPU. If this cell fails, STOP and report back —
# do not continue with unpinned versions (the baseline-reproduction gate depends on this environment).
# Colab's preinstalled torchvision/torchaudio are built against a different torch and are
# not used by PromptShield; leaving them installed breaks Transformers with
# 'operator torchvision::nms does not exist'. Remove them before AND after the pinned
# install, then verify the final state in this same cell.
# Pip may still warn about unrelated preinstalled Colab packages; the explicit checks below
# and Cell 3 are authoritative.
ENV_GUARD_BUILD = 'hardneg-env-v3'
%pip uninstall -y torchvision torchaudio
%pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cpu
%pip install -q transformers==4.49.0 sentencepiece==0.2.1 pyarrow==22.0.0 scikit-learn==1.8.0 joblib==1.5.3
%pip uninstall -y torchvision torchaudio

import importlib.metadata
import importlib.util
import sys

for package in ('torchvision', 'torchaudio'):
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    try:
        spec = importlib.util.find_spec(package)
    except (AttributeError, ImportError, ValueError) as error:
        spec = f'<broken import spec: {error}>'
    loaded = sorted(
        name for name in sys.modules
        if name == package or name.startswith(package + '.')
    )
    assert installed is None and spec is None and not loaded, (
        f'STOP: {package} survived Cell 2 (installed={installed!r}, spec={spec!r}, loaded={loaded}). '
        'Restart with Runtime -> Disconnect and delete runtime, then rerun Cell 2. '
        'Do not install TorchVision; do not change the torch pin.'
    )
CELL2_INSTALL_OK = True
print('Cell 2 verified: torchvision/torchaudio absent; pinned install complete; guard build', ENV_GUARD_BUILD)

In [ ]:
# CELL 3 — environment verification: pins, CPU-only, media guard, offline mode, import location, RAM gate.
import importlib.metadata
import importlib.util
import os
import sys

assert globals().get('CELL2_INSTALL_OK') is True, (
    'STOP: Cell 2 did not complete in this runtime. Restart with Runtime -> Disconnect and delete '
    'runtime, rerun Cell 2, then rerun this cell.'
)
assert globals().get('ENV_GUARD_BUILD') == 'hardneg-env-v3', (
    'STOP: stale notebook or stale runtime. Cell 2 must print guard build hardneg-env-v3. '
    'Upload the current notebook, restart the runtime, rerun Cell 2, then rerun this cell.'
)

# No external model or dataset downloads (the checkpoint is local and complete).
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from importlib.metadata import version

expected_versions = {
    'torch': '2.6.0+cpu',
    'transformers': '4.49.0',
    'sentencepiece': '0.2.1',
    'pyarrow': '22.0.0',
    'scikit-learn': '1.8.0',
    'joblib': '1.5.3',
}
found = {}
for dist in expected_versions:
    try:
        found[dist] = version(dist)
    except Exception as error:
        found[dist] = f'MISSING ({error})'
mismatch = {k: {'expected': v, 'found': found[k]} for k, v in expected_versions.items() if found[k] != v}
assert not mismatch, f'STOP: dependency mismatch {mismatch}. Do not train; report back.'

import torch
assert torch.__version__ == expected_versions['torch'], (
    f'STOP: imported torch {torch.__version__} != pinned {expected_versions["torch"]}. '
    'Restart the runtime (Runtime -> Disconnect and delete runtime), rerun Cell 2, then rerun this cell.'
)
assert not torch.cuda.is_available(), (
    'GPU is visible. The frozen plan runs on CPU: Runtime -> Change runtime type -> CPU, then restart and rerun.'
)

# --- BEGIN COLAB MEDIA GUARD ---
# This block is also executed by tests/test_hard_negative_colab_environment.py.
import importlib.metadata
import importlib.util
import sys

MEDIA_PACKAGES = ('torchvision', 'torchaudio')
ENV_GUARD_BUILD = 'hardneg-env-v3'

def media_presence():
    presence = {}
    for package in MEDIA_PACKAGES:
        try:
            distribution = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            distribution = None
        try:
            spec = importlib.util.find_spec(package)
        except (AttributeError, ImportError, ValueError) as error:
            spec = f'<broken import spec: {error}>'
        loaded = sorted(
            name for name in sys.modules
            if name == package or name.startswith(package + '.')
        )
        presence[package] = {
            'distribution': distribution,
            'spec': None if spec is None else getattr(spec, 'origin', str(spec)),
            'spec_found': spec is not None,
            'loaded_modules': loaded,
        }
    return presence

def assert_media_absent(stage):
    found = {
        package: state
        for package, state in media_presence().items()
        if state['distribution'] is not None or state['spec_found'] or state['loaded_modules']
    }
    assert not found, (
        f'STOP [{ENV_GUARD_BUILD}] at {stage}: torchvision/torchaudio are still present {found}. '
        'This is a stale runtime, a failed uninstall, or an old Cell 2. Restart with Runtime -> '
        'Disconnect and delete runtime, upload the current notebook, rerun Cell 2, then rerun this cell. '
        'Do not install TorchVision; do not change the torch pin.'
    )

def assert_transformers_media_state(transformers_module, image_utils_module):
    utils_module = getattr(transformers_module, 'utils', None)
    def media_flag(package):
        for module in (transformers_module, utils_module):
            checker = getattr(module, f'is_{package}_available', None) if module is not None else None
            if callable(checker):
                return bool(checker())
        return False
    available = {package: media_flag(package) for package in MEDIA_PACKAGES}
    stale_refs = [
        name for name in ('torchvision_io', 'pil_torch_interpolation_mapping')
        if hasattr(image_utils_module, name)
    ]
    assert not any(available.values()) and not stale_refs, (
        f'STOP [{ENV_GUARD_BUILD}]: Transformers cached optional media as available {available} '
        f'or image_utils holds stale references {stale_refs}. The interpreter is stale even if the '
        'media packages were uninstalled. Restart with Runtime -> Disconnect and delete runtime, '
        'rerun Cell 2, then rerun this cell. Do not install TorchVision; do not change the torch pin.'
    )
# --- END COLAB MEDIA GUARD ---

assert_media_absent('before importing Transformers')
try:
    import transformers
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DebertaV2Config,
        DebertaV2ForSequenceClassification,
        DebertaV2Tokenizer,
    )
except Exception as error:
    raise AssertionError(
        f'STOP [{ENV_GUARD_BUILD}]: Transformers/DeBERTa import failed before Gate 0: '
        f'{type(error).__name__}: {error}. Restart with Runtime -> Disconnect and delete runtime, '
        'rerun Cell 2, then rerun this cell. Do not change the torch or Transformers pin.'
    ) from error
assert transformers.__version__ == expected_versions['transformers']
assert_transformers_media_state(transformers, sys.modules.get('transformers.image_utils'))
try:
    import transformers.image_utils as transformers_image_utils
    import transformers.processing_utils  # imports image_utils and its optional torchvision branch
except Exception as error:
    raise AssertionError(
        f'STOP [{ENV_GUARD_BUILD}]: Transformers optional vision import failed before Gate 0: '
        f'{type(error).__name__}: {error}. This is the TorchVision/PyTorch mismatch. '
        'Restart with Runtime -> Disconnect and delete runtime, rerun Cell 2, then rerun this cell. '
        'Do not install TorchVision; do not change the torch pin.'
    ) from error
assert_transformers_media_state(transformers, transformers_image_utils)
assert_media_absent('after importing Transformers/DeBERTa')

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR / 'src'))
import promptshield
assert str(WORKDIR) in str(promptshield.__file__), f'Wrong promptshield imported: {promptshield.__file__}'

from promptshield import hard_negative_experiment as hne

# Same pre-registered precondition as the local run: >= 3.5 GiB free (fails closed).
free_bytes = hne.check_free_ram(hne.free_physical_memory_bytes())

CELL3_GUARD_OK = True
import platform
print({
    'guard_build': ENV_GUARD_BUILD,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': version('transformers'),
    'free_ram_gib': round(free_bytes / 2**30, 2),
    'cuda': torch.cuda.is_available(),
    'device': 'cpu',
    'torchvision': 'absent',
    'torchaudio': 'absent',
    'transformers_image_utils': 'imported',
    'transformers_deberta_classes': 'imported',
    'promptshield': str(promptshield.__file__),
    'working_directory': os.getcwd(),
})
print('environment verified; guard build', ENV_GUARD_BUILD)

In [ ]:
# CELL 4 — GATE 0 (read-only): reproduce the frozen baseline audit BEFORE any training.
# Loads the validation split through the audit tool, writes nothing, trains nothing.
# If the confusion matrix differs, this cell fails: STOP, do not train, report back.
assert globals().get('ENV_GUARD_BUILD') == 'hardneg-env-v3', (
    'STOP: Cell 2/3 did not run the current guard in this runtime. Restart with Runtime -> '
    'Disconnect and delete runtime, upload the current notebook, rerun Cells 2 and 3, then retry.'
)
assert globals().get('CELL2_INSTALL_OK') is True, (
    'STOP: Cell 2 did not complete in this runtime. Rerun Cell 2 and Cell 3 before Gate 0.'
)
assert globals().get('CELL3_GUARD_OK') is True, (
    'STOP: Cell 3 environment gate did not complete in this runtime. Rerun Cell 3 before Gate 0.'
)
from promptshield.deberta_decision_audit import audit_validation

baseline_audit = audit_validation(
    hne.VALIDATION_PATH, hne.BASELINE_DIR, threshold=hne.DEFAULT_THRESHOLD, batch_size=16
)
cm = baseline_audit['comparison']['threshold']['metrics']['confusion_matrix']
assert cm == hne.EXPECTED_BASELINE_CM, (
    f'ABORT: baseline confusion matrix {cm} != frozen {hne.EXPECTED_BASELINE_CM}. '
    'No training may run. Report this environment mismatch back.'
)
print('Gate 0 passed: baseline audit reproduced', cm, 'at threshold 0.90 (validation split only)')

## The single run

The next cell executes `hard_negative_experiment.main()` — **the entire frozen pipeline, exactly once:**

1. RAM precondition (>= 3.5 GiB) — aborts otherwise.
2. **Fresh baseline audit** must reproduce `[[365, 42], [9, 525]]` exactly — aborts otherwise (this is the same check as Gate 0 above, re-run as the official gate).
3. `train.parquet` and baseline checkpoint hash verification; build of the 138 hard negatives + 138 malicious anchors + 138 easy-benign anchors (414 rows) with all frozen exclusion rules; counts deviating from the plan stop the run.
4. **One training run** (warm start from the frozen checkpoint, frozen input embeddings, AdamW lr 1e-5, weight decay 0.01, 3 epochs, batch 4, max_length 128, seed 42, CPU) into the NEW directory `models/deberta-v3-small-hardneg-v1/`. The baseline directory is never written.
5. Candidate audit with the identical protocol (threshold 0.90), plus the 4/4 positive control.
6. Acceptance criteria **G1–G6** applied unchanged → retain or reject. A failed criterion is a valid, final result: document it and stop — no retries, no tuning.

**Do not run this cell unless you intend to consume the single pre-registered run.** Outputs: aggregate JSON on screen and `reports/hard-negative-mitigation-<ts>.{json,md}`.

In [ ]:
# CELL 5 — THE SINGLE PRE-REGISTERED RUN (training happens here and only here).
assert globals().get('ENV_GUARD_BUILD') == 'hardneg-env-v3' and globals().get('CELL3_GUARD_OK') is True, (
    'STOP: the Cell 2/3 environment gate did not complete in this runtime. Do not train; '
    'restart the runtime, rerun Cells 2-4, and only then retry.'
)
assert not (WORKDIR / 'models' / 'deberta-v3-small-hardneg-v1').exists(), (
    'A candidate artifact already exists: the plan allows exactly ONE training run. '
    'Do not rerun. Collect the existing outputs and stop.'
)
(WORKDIR / 'reports').mkdir(exist_ok=True)
(WORKDIR / 'data').mkdir(exist_ok=True)

from promptshield.hard_negative_experiment import main

exit_code = main(['--reports-dir', 'reports'])
if exit_code != 0:
    raise RuntimeError(
        'Run BLOCKED by a frozen-plan gate (JSON above). STOP HERE: save the output, '
        'do not retry, do not tune, do not change thresholds, and report the result back.'
    )
print('Run completed. Decision and metrics are in the JSON above and in reports/.')

In [ ]:
# CELL 6 — after the run: baseline integrity re-check + aggregate report summary (no raw prompts).
import json

for name, expected in EXPECTED_BASELINE_FILES.items():
    actual = sha256(WORKDIR / 'models/deberta-v3-small' / name)
    assert actual == expected, f'BASELINE MODIFIED DURING THE RUN: {name}'
print('baseline checkpoint verified unchanged after the experiment')

report_paths = sorted((WORKDIR / 'reports').glob('hard-negative-mitigation-*.json'))
assert report_paths, 'No report was produced.'
report = json.loads(report_paths[-1].read_text(encoding='utf-8'))
assert report['test_split_loaded'] is False
assert report['single_run'] is True

print(json.dumps({
    'decision': report['acceptance_criteria']['decision'],
    'gates': report['gates'],
    'criteria': [
        {key: check[key] for key in ('id', 'observed', 'baseline', 'required', 'passed')}
        for check in report['acceptance_criteria']['checks']
    ],
    'baseline_confusion_matrix': report['evaluation']['baseline_threshold_metrics']['confusion_matrix'],
    'candidate_confusion_matrix': report['evaluation']['candidate_threshold_metrics']['confusion_matrix'],
    'positive_control': report['evaluation']['positive_control'],
    'report_file': report_paths[-1].name,
    'test_split_loaded': report['test_split_loaded'],
    'raw_prompts_in_report': report['integrity']['raw_prompts_in_report'],
}, indent=2))
print('If decision == reject: keep the artifact labeled "rejected candidate"; the baseline remains the only presented model.')

## Returning results to the local project

The package below contains **only**:

- `models/deberta-v3-small-hardneg-v1/` — the candidate checkpoint (saved separately; the baseline was never written),
- `data/hardneg-manifest.json` — metadata only (ids, hashes, counts; **no prompt text**),
- `reports/hard-negative-mitigation-<ts>.{json,md}` — aggregate metrics and metadata only.

It never contains `test.parquet`, training data, raw prompts, or anything from the repository beyond the above.

**How to transfer back:** copy the zip to Google Drive with the optional cell below (recommended), or use
`google.colab.files.download(zip_path)` for a direct browser download. On the local machine, place the
candidate under `models/deberta-v3-small-hardneg-v1/`, the manifest under `data/`, and the reports under
`reports/` — never over `models/deberta-v3-small/`. Keep the candidate labeled with the report's decision
(retain or reject) until the project owner reviews it. **No deployment, no performance claims.**

In [ ]:
# CELL 7 — package the return artifacts (run only after the experiment finished).
import datetime
import zipfile

stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_zip = Path(f'/content/promptshield-hardneg-artifacts-{stamp}.zip')

targets = [
    WORKDIR / 'models' / 'deberta-v3-small-hardneg-v1',
    WORKDIR / 'data' / 'hardneg-manifest.json',
    WORKDIR / 'reports',
]
present = [target for target in targets if target.exists()]
assert present, 'Nothing to package: did the run complete?'

with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for target in present:
        if target.is_dir():
            for file in sorted(target.rglob('*')):
                if file.is_file():
                    archive.write(file, file.relative_to(WORKDIR))
        else:
            archive.write(target, target.relative_to(WORKDIR))

with zipfile.ZipFile(out_zip) as archive:
    names = archive.namelist()
assert not any('test.parquet' in name for name in names)
print(out_zip, f'{out_zip.stat().st_size} bytes')
print('contents (aggregate):', names if len(names) <= 40 else f'{len(names)} files')

In [ ]:
# CELL 8 (optional) — copy the artifact zip to your Google Drive for download to the local project.
from google.colab import drive

drive.mount('/content/drive')
import shutil
shutil.copy(out_zip, '/content/drive/MyDrive/' + out_zip.name)
print('saved to MyDrive: /content/drive/MyDrive/' + out_zip.name)
# Alternative (direct browser download): from google.colab import files; files.download(str(out_zip))